# BiomeGPT Reproduction (Colab Notebook)

This notebook is a practical reproduction scaffold based on your `dataset_v3` files.

What this notebook does:
1. Reports mean, median, 25th percentile, and 75th percentile for non-zero species per sample on gut + non-gut dataset.
2. Stage 1 pretraining (gut + non-gut) for 30 epochs.
3. Stage 2 domain adaptation (gut-only) for 3-5 extra epochs.

This version follows your requested training details:
- masked abundance-bin prediction with **MSE loss**
- paper-style **attention masking** in pretraining

Recommended runtime in Colab: `GPU` (A100/L4/T4).

## Cell 1 - Install dependencies
**What this cell does:** installs the Python packages needed for loading CSV/ZIP data and training with PyTorch.

In [1]:
# If packages already exist in Colab runtime, this finishes quickly.
!pip -q install pandas numpy torch

## Cell 2 - Imports and global setup
**What this cell does:** imports libraries, sets random seed for reproducibility, and selects CPU/GPU device.

In [3]:
import os
import json
import random
import zipfile
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [4]:
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

NVIDIA A100-SXM4-40GB


## Cell 3 - Set paths and training hyperparameters
**What this cell does:** defines where your data files live and sets key settings (32 bins, 30 + extra epochs).

If you put files on Google Drive, set `DATA_DIR` accordingly.

In [ ]:
# Option A: files are uploaded to current Colab working directory
DATA_DIR = Path(".").resolve()

# Option B: files are in Google Drive (uncomment and edit)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/your_folder_here')

PHASE1_ABUND = DATA_DIR / "abund_pretraining_phase1_gut_and_nongut.csv.zip"
PHASE1_META  = DATA_DIR / "meta_pretraining_phase1_gut_and_nongut.csv"
PHASE2_ABUND = DATA_DIR / "abund_pretraining_phase2_gut.csv.zip"
PHASE2_META  = DATA_DIR / "meta_pretraining_phase2_gut.csv"

OUTPUT_DIR = DATA_DIR / "outputs_biomegpt_colab"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Teacher-requested defaults
NUM_BINS = 32
EPOCHS_STAGE1 = 30
EPOCHS_STAGE2 = 5   # Change to 3, 4, or 5 as needed
MASK_RATIO = 0.25
BATCH_SIZE = 8      # Increase if your GPU memory allows
LR = 1e-4
MIXED_PRECISION = True

print("DATA_DIR:", DATA_DIR)

## Cell 4 - Data loading and descriptive statistics
**What this cell does:**
- loads abundance + metadata tables,
- aligns sample IDs,
- computes the required statistics for non-zero species counts.

In [ ]:
@dataclass
class NonZeroStats:
    mean: float
    median: float
    p25: float
    p75: float


def load_csv_or_zip(path: Path, index_col: int = 0) -> pd.DataFrame:
    # Load a CSV directly, or the first CSV file inside a ZIP
    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path, "r") as zf:
            members = [m for m in zf.namelist() if (not m.endswith("/")) and (not m.startswith("__MACOSX/"))]
            if not members:
                raise RuntimeError(f"No CSV found inside ZIP: {path}")
            members = sorted(members, key=lambda x: (not x.lower().endswith(".csv"), x))
            with zf.open(members[0]) as f:
                return pd.read_csv(f, index_col=index_col)
    return pd.read_csv(path, index_col=index_col)


def align_abund_meta(abund: pd.DataFrame, meta: pd.DataFrame):
    # Keep only sample IDs that appear in both abundance and metadata
    common = abund.index.intersection(meta.index)
    if len(common) == 0:
        raise RuntimeError("No overlapping sample IDs between abundance and metadata")
    return abund.loc[common], meta.loc[common]


def compute_nonzero_stats(abund: pd.DataFrame) -> NonZeroStats:
    # Compute mean/median/p25/p75 of non-zero species count per sample
    nonzero_counts = (abund > 0).sum(axis=1)
    return NonZeroStats(
        mean=float(nonzero_counts.mean()),
        median=float(nonzero_counts.median()),
        p25=float(nonzero_counts.quantile(0.25)),
        p75=float(nonzero_counts.quantile(0.75)),
    )


# Load phase1 (gut + non-gut), then compute teacher-required stats
phase1_abund = load_csv_or_zip(PHASE1_ABUND, index_col=0)
phase1_meta = load_csv_or_zip(PHASE1_META, index_col=0)
phase1_abund, phase1_meta = align_abund_meta(phase1_abund, phase1_meta)

stats = compute_nonzero_stats(phase1_abund)
print("[Phase1 gut+non-gut] non-zero species per sample")
print(f"Mean:   {stats.mean:.4f}")
print(f"Median: {stats.median:.4f}")
print(f"P25:    {stats.p25:.4f}")
print(f"P75:    {stats.p75:.4f}")

with open(OUTPUT_DIR / "phase1_nonzero_stats.json", "w", encoding="utf-8") as f:
    json.dump(asdict(stats), f, indent=2)

print("Saved stats JSON ->", OUTPUT_DIR / "phase1_nonzero_stats.json")

## Cell 5 - Abundance binning helpers
**What this cell does:** converts continuous abundance profile into rank-based bins (`0..32`) per sample.

- `0` means species is absent in that sample.
- `1..32` represent increasing abundance rank.

In [ ]:
def rank_to_bins(nonzero_values: np.ndarray, num_bins: int) -> np.ndarray:
    # Convert non-zero abundances to bins in [1, num_bins], highest abundance gets highest bin
    n = nonzero_values.shape[0]
    if n == 0:
        return np.empty((0,), dtype=np.int64)

    order = np.argsort(-nonzero_values)
    bins = np.zeros(n, dtype=np.int64)

    if n >= num_bins:
        ranks = np.arange(n, dtype=np.float64)
        assigned = num_bins - np.floor(ranks * num_bins / n).astype(np.int64)
        assigned = np.clip(assigned, 1, num_bins)
    else:
        if n == 1:
            assigned = np.array([num_bins], dtype=np.int64)
        else:
            ranks = np.arange(n, dtype=np.float64)
            assigned = np.round((n - 1 - ranks) * (num_bins - 1) / (n - 1)).astype(np.int64) + 1
            assigned = np.clip(assigned, 1, num_bins)

    bins[order] = assigned
    return bins


def abundance_to_binned_matrix(abund: pd.DataFrame, num_bins: int) -> np.ndarray:
    # Convert abundance DataFrame [samples, species] into integer bin matrix [samples, species]
    arr = abund.to_numpy(dtype=np.float32)
    out = np.zeros_like(arr, dtype=np.int64)

    for i in range(arr.shape[0]):
        row = arr[i]
        nz_idx = np.flatnonzero(row > 0)
        if nz_idx.size == 0:
            continue
        out[i, nz_idx] = rank_to_bins(row[nz_idx], num_bins)

    return out

## Cell 6 - Dataset and model definition (with paper-style attention mask)
**What this cell does:** defines:
- a PyTorch dataset for binned species matrix,
- a BiomeGPT-style transformer model,
- paper-style attention masking rules in pretraining.

In [ ]:
class SpeciesBinDataset(Dataset):
    # Dataset that returns one sample's binned species vector at a time
    def __init__(self, binned_matrix: np.ndarray):
        self.x = torch.from_numpy(binned_matrix).long()

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx]


class BiomeGPTSpeciesModel(nn.Module):
    # Minimal species-only pretraining model inspired by BiomeGPT description
    def __init__(
        self,
        num_species: int,
        num_bins: int,
        d_model: int = 512,
        nhead: int = 8,
        num_layers: int = 8,
        ff_dim: int = 512,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.num_species = num_species
        self.num_bins = num_bins
        self.nhead = nhead

        # species IDs: 0 is <cls>, 1..num_species are species tokens
        self.species_emb = nn.Embedding(num_species + 1, d_model)

        # abundance-bin embedding via small MLP
        self.abund_mlp = nn.Sequential(
            nn.Linear(1, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model),
        )

        self.ln_species = nn.LayerNorm(d_model)
        self.ln_abund = nn.LayerNorm(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            activation="relu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Regression head: predict masked abundance-bin value (MSE objective)
        self.head = nn.Sequential(
            nn.Linear(d_model, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )

        species_ids = torch.arange(1, num_species + 1, dtype=torch.long).unsqueeze(0)
        self.register_buffer("species_ids", species_ids, persistent=False)

    def _build_paper_attention_mask(self, bins: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # Build per-sample attention mask with rules:
        # 1) zero-abundance species do not participate in cross-token attention
        # 2) unmasked species attend to unmasked species
        # 3) masked species attend to unmasked species + themselves
        bsz, seq_len = bins.shape
        L = seq_len + 1  # +1 for <cls>
        device = bins.device

        nonzero = bins > 0
        unmasked = nonzero & (~mask)
        masked = nonzero & mask

        allow = torch.zeros((bsz, L, L), dtype=torch.bool, device=device)

        # <cls> token sees itself and unmasked species
        allow[:, 0, 0] = True
        allow[:, 0, 1:] = unmasked

        # species->species attention block
        unmasked_q = unmasked.unsqueeze(2)  # [B, S, 1]
        masked_q = masked.unsqueeze(2)      # [B, S, 1]
        unmasked_k = unmasked.unsqueeze(1)  # [B, 1, S]

        species_allow = (unmasked_q & unmasked_k) | (masked_q & unmasked_k)

        # Keep self-attention for every species row to avoid fully-masked rows
        eye = torch.eye(seq_len, dtype=torch.bool, device=device).unsqueeze(0)
        species_allow = species_allow | eye

        allow[:, 1:, 1:] = species_allow

        # MultiheadAttention: bool mask, True means disallow
        disallow = ~allow
        disallow = (
            disallow.unsqueeze(1)
            .expand(bsz, self.nhead, L, L)
            .reshape(bsz * self.nhead, L, L)
        )
        return disallow

    def forward(self, bins: torch.Tensor, mask: torch.Tensor):
        # bins: [B, S], values 0..num_bins; mask: [B, S] True for masked positions
        bsz, seq_len = bins.shape
        if seq_len != self.num_species:
            raise ValueError(f"Expected {self.num_species} species, got {seq_len}")

        # Replace masked bins with 0 for model input
        input_bins = bins.clone()
        input_bins[mask] = 0

        # Add a <cls> token to the front
        cls_id = torch.zeros((bsz, 1), dtype=torch.long, device=bins.device)
        species_ids = self.species_ids.expand(bsz, -1)
        all_species_ids = torch.cat([cls_id, species_ids], dim=1)

        cls_bin = torch.zeros((bsz, 1), dtype=torch.long, device=bins.device)
        all_bins = torch.cat([cls_bin, input_bins], dim=1)

        # Input token = species embedding + abundance embedding
        species_tok = self.ln_species(self.species_emb(all_species_ids))
        abund_tok = self.ln_abund(self.abund_mlp((all_bins.float() / max(self.num_bins, 1)).unsqueeze(-1)))
        x = species_tok + abund_tok

        # Apply paper-style attention mask
        attn_mask = self._build_paper_attention_mask(bins=bins, mask=mask)
        h = self.encoder(x, mask=attn_mask)

        # Regress bin values per species token
        preds = self.head(h[:, 1:, :]).squeeze(-1)  # [B, S]
        return preds

## Attention Mask Sanity Test (Recommended)
**What this cell does:**
- builds a tiny toy sample,
- materializes the paper-style attention mask,
- asserts the 3 rules directly:
  1) zero-abundance species do not participate in cross-token attention,
  2) unmasked species can attend to unmasked species,
  3) masked species can attend to unmasked species and themselves,
- prints a readable attention-allow matrix (`1=can attend`, `0=blocked`).


In [ ]:
# This is a lightweight unit-test style sanity check before training.

def get_allow_matrix_for_sample(model, bins, mask, sample_idx=0, head_idx=0):
    disallow = model._build_paper_attention_mask(bins=bins, mask=mask)
    idx = sample_idx * model.nhead + head_idx
    allow = (~disallow[idx]).detach().cpu().numpy().astype(int)
    return allow

# Toy example with both zero, masked, and unmasked non-zero species
# species indices: 0..5
# bins: [0, 5, 3, 0, 2, 7]
# masked non-zero: species 1 and 4
bins_toy = torch.tensor([[0, 5, 3, 0, 2, 7]], dtype=torch.long, device=DEVICE)
mask_toy = torch.tensor([[False, True, False, False, True, False]], dtype=torch.bool, device=DEVICE)

allow = get_allow_matrix_for_sample(model, bins_toy, mask_toy, sample_idx=0, head_idx=0)

# Token positions in attention matrix: 0 is <cls>, species j is token (j+1)
def tok(j):
    return j + 1

nonzero = [1, 2, 4, 5]
masked = [1, 4]
unmasked = [2, 5]
zeros = [0, 3]

# Rule 1: zero-abundance species should not have cross-token attention
for z in zeros:
    zt = tok(z)
    for col in range(allow.shape[1]):
        if col != zt:
            assert allow[zt, col] == 0, f"Rule1 failed: zero species sp{z} should not attend to token {col}"
    for row in range(allow.shape[0]):
        if row != zt:
            assert allow[row, zt] == 0, f"Rule1 failed: token {row} should not attend to zero species sp{z}"

# Rule 2: unmasked species can attend to unmasked species
for u in unmasked:
    ut = tok(u)
    for v in unmasked:
        vt = tok(v)
        assert allow[ut, vt] == 1, f"Rule2 failed: unmasked sp{u} should attend unmasked sp{v}"

# Rule 3: masked species can attend to unmasked species and themselves (but not other masked)
for m in masked:
    mt = tok(m)
    # self
    assert allow[mt, mt] == 1, f"Rule3 failed: masked sp{m} should attend itself"
    # unmasked
    for u in unmasked:
        ut = tok(u)
        assert allow[mt, ut] == 1, f"Rule3 failed: masked sp{m} should attend unmasked sp{u}"
    # other masked
    for m2 in masked:
        if m2 == m:
            continue
        mt2 = tok(m2)
        assert allow[mt, mt2] == 0, f"Rule3 failed: masked sp{m} should NOT attend masked sp{m2}"

print("PASS: attention rules match expected behavior on toy example")

labels = ["<cls>"] + [f"sp{i}" for i in range(bins_toy.shape[1])]
allow_df = pd.DataFrame(allow, index=labels, columns=labels)
print("
Attention allow matrix (1=can attend, 0=blocked):")
display(allow_df)


## Cell 7 - Masking + training functions
**What this cell does:**
- randomly masks ~25% non-zero species per sample,
- trains one stage (multiple epochs),
- uses **MSE loss only on masked non-zero species**,
- saves checkpoint.

In [ ]:
def build_mask(batch_bins: torch.Tensor, mask_ratio: float, generator: torch.Generator) -> torch.Tensor:
    # Mask a fraction of non-zero species in each sample
    bsz, seq_len = batch_bins.shape
    mask = torch.zeros((bsz, seq_len), dtype=torch.bool, device=batch_bins.device)

    for i in range(bsz):
        nz_idx = torch.nonzero(batch_bins[i] > 0, as_tuple=False).squeeze(-1)
        if nz_idx.numel() == 0:
            continue
        k = max(1, int(round(mask_ratio * nz_idx.numel())))
        perm = torch.randperm(nz_idx.numel(), generator=generator, device=batch_bins.device)
        mask[i, nz_idx[perm[:k]]] = True

    return mask


def train_stage(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    epochs: int,
    mask_ratio: float,
    stage_name: str,
    seed: int,
    mixed_precision: bool = True,
):
    # Train one stage (stage1 or stage2)
    scaler = torch.amp.GradScaler("cuda", enabled=(mixed_precision and device.type == "cuda"))
    generator = torch.Generator(device=device.type if device.type in {"cpu", "cuda"} else "cpu")
    generator.manual_seed(seed)

    model.train()
    for epoch in range(1, epochs + 1):
        running_loss = 0.0
        steps = 0

        for batch_bins in loader:
            batch_bins = batch_bins.to(device, non_blocking=True)
            mask = build_mask(batch_bins, mask_ratio=mask_ratio, generator=generator)
            valid = mask & (batch_bins > 0)

            if valid.sum().item() == 0:
                continue

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(mixed_precision and device.type == "cuda")):
                preds = model(batch_bins, mask)
                # MSE only on masked non-zero species
                loss = F.mse_loss(preds[valid], batch_bins[valid].float())

            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            running_loss += float(loss.detach().cpu().item())
            steps += 1

        epoch_loss = running_loss / max(steps, 1)
        print(f"[{stage_name}] epoch {epoch:02d}/{epochs} - loss: {epoch_loss:.6f}")


def save_checkpoint(path: Path, model: nn.Module, optimizer: torch.optim.Optimizer, extra: dict):
    # Save model + optimizer state for resuming or later evaluation
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "extra": extra,
    }
    torch.save(payload, path)
    print("Saved:", path)

## Cell 8 - Prepare stage1/stage2 matrices and loaders
**What this cell does:**
- loads phase2 gut-only data,
- aligns species vocabulary with stage1,
- converts both stages to binned matrices,
- creates DataLoader objects.

In [ ]:
# Load phase2 (gut-only)
phase2_abund = load_csv_or_zip(PHASE2_ABUND, index_col=0)
phase2_meta = load_csv_or_zip(PHASE2_META, index_col=0)
phase2_abund, phase2_meta = align_abund_meta(phase2_abund, phase2_meta)

# Keep stage1 species order to make input dimension consistent between stages
phase1_species = phase1_abund.columns.tolist()
phase2_abund = phase2_abund.reindex(columns=phase1_species, fill_value=0.0)

print("phase1_abund shape:", phase1_abund.shape)
print("phase2_abund shape (reindexed):", phase2_abund.shape)

# Convert abundance to rank bins
phase1_bins = abundance_to_binned_matrix(phase1_abund, NUM_BINS)
phase2_bins = abundance_to_binned_matrix(phase2_abund, NUM_BINS)

print("phase1_bins shape:", phase1_bins.shape)
print("phase2_bins shape:", phase2_bins.shape)

stage1_loader = DataLoader(
    SpeciesBinDataset(phase1_bins),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)

stage2_loader = DataLoader(
    SpeciesBinDataset(phase2_bins),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)

## Cell 9 - Build model and optimizer
**What this cell does:** initializes transformer model and AdamW optimizer.

In [ ]:
model = BiomeGPTSpeciesModel(
    num_species=phase1_bins.shape[1],
    num_bins=NUM_BINS,
    d_model=512,
    nhead=8,
    num_layers=8,
    ff_dim=512,
    dropout=0.1,
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
print("Model initialized")

## Cell 10 - Stage 1 pretraining (gut + non-gut, 30 epochs)
**What this cell does:** runs stage1 training and saves `checkpoint_stage1.pt`.

In [ ]:
train_stage(
    model=model,
    loader=stage1_loader,
    optimizer=optimizer,
    device=DEVICE,
    epochs=EPOCHS_STAGE1,
    mask_ratio=MASK_RATIO,
    stage_name="Stage1 pretrain (gut+non-gut)",
    seed=SEED,
    mixed_precision=MIXED_PRECISION,
)

save_checkpoint(
    OUTPUT_DIR / "checkpoint_stage1.pt",
    model,
    optimizer,
    {
        "num_bins": NUM_BINS,
        "epochs_stage1": EPOCHS_STAGE1,
        "mask_ratio": MASK_RATIO,
        "batch_size": BATCH_SIZE,
        "lr": LR,
    },
)

## Cell 11 - Stage 2 domain adaptation (gut-only, 3-5 epochs)
**What this cell does:** continues training from stage1 weights using only gut samples and saves `checkpoint_stage2.pt`.

In [ ]:
train_stage(
    model=model,
    loader=stage2_loader,
    optimizer=optimizer,
    device=DEVICE,
    epochs=EPOCHS_STAGE2,
    mask_ratio=MASK_RATIO,
    stage_name="Stage2 domain-adapt (gut-only)",
    seed=SEED + 1,
    mixed_precision=MIXED_PRECISION,
)

save_checkpoint(
    OUTPUT_DIR / "checkpoint_stage2.pt",
    model,
    optimizer,
    {
        "num_bins": NUM_BINS,
        "epochs_stage2": EPOCHS_STAGE2,
        "mask_ratio": MASK_RATIO,
        "batch_size": BATCH_SIZE,
        "lr": LR,
    },
)

## Cell 12 - Final summary report
**What this cell does:** prints a compact report you can paste into your progress notes.

In [ ]:
summary = {
    "teacher_required_stats_phase1": asdict(stats),
    "training_setup": {
        "num_bins": NUM_BINS,
        "epochs_stage1": EPOCHS_STAGE1,
        "epochs_stage2": EPOCHS_STAGE2,
        "mask_ratio": MASK_RATIO,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "device": str(DEVICE),
    },
    "artifacts": {
        "stats_json": str(OUTPUT_DIR / "phase1_nonzero_stats.json"),
        "checkpoint_stage1": str(OUTPUT_DIR / "checkpoint_stage1.pt"),
        "checkpoint_stage2": str(OUTPUT_DIR / "checkpoint_stage2.pt"),
    },
}
print(json.dumps(summary, indent=2))